In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate langdetect requests

!pip -q install -U pageindex


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, pickle
from collections import Counter
from typing import List, Dict, Any

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
PAGEINDEX_API_KEY = userdata.get('PAGEINDEX_API_KEY')
print(PAGEINDEX_API_KEY)

604fe7e3a26e4d3487b3f99c35b23361


In [ ]:
CORPUS_ROOT = "/content/drive/MyDrive/ColabNotebooks/corpus"

# Embedding model
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# Explicit Nepali author registry
NEPALI_AUTHORS = {
    "LaxmiPrasadDevkota",
}

# Output artifacts
OUT_DIR = "/content/model"
os.makedirs(OUT_DIR, exist_ok=True)

CHUNKS_PKL_PATH = os.path.join(OUT_DIR, "chunks.pkl")
FAISS_INDEX_PATH = os.path.join(OUT_DIR, "faiss.index")


In [ ]:
DEVANAGARI_RE = re.compile(r"[\u0900-\u097F]")

def normalize_ws(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip()

def is_nepali(text: str) -> bool:
    return bool(DEVANAGARI_RE.search(text))

def sent_split(text: str, lang: str) -> List[str]:
    """
    Lightweight sentence splitter for EN/NE without regex lookbehind.
    Avoids splitting inside common English abbreviations by temporary masking.
    """
    text = normalize_ws(text)
    if not text:
        return []

    if lang == "ne":
        parts = re.split(r"[।!?]+", text)
        return [p.strip() for p in parts if p and p.strip()]

    # EN: mask common abbreviations so we don't split on their periods
    abbreviations = ["Mr.", "Mrs.", "Ms.", "Dr.", "Prof.", "Sr.", "Jr.", "St.", "vs.", "e.g.", "i.e."]
    for abbr in abbreviations:
        text = text.replace(abbr, abbr.replace(".", "<DOT>"))

    # split on sentence-ending punctuation
    parts = re.split(r"[!?]+|\.+", text)
    parts = [p.strip() for p in parts if p and p.strip()]

    # unmask
    parts = [p.replace("<DOT>", ".") for p in parts]
    return parts


# Token-ish chunk targets (post-PageIndex passages)
MIN_TOKENS_PER_CHUNK = 200
MAX_TOKENS_PER_CHUNK = 350
CHUNK_OVERLAP_TOKENS = 40

def _est_tokens_from_words(n_words: int, lang: str) -> int:
    # heuristic; stable enough for chunk sizing without a tokenizer dependency
    return int(n_words * (1.30 if lang == "ne" else 1.33))

def chunkify(text: str,
             lang: str,
             min_tokens: int = MIN_TOKENS_PER_CHUNK,
             max_tokens: int = MAX_TOKENS_PER_CHUNK,
             overlap_tokens: int = CHUNK_OVERLAP_TOKENS) -> List[str]:
    """
    Sentence-packed chunking aiming for ~200–350 tokens with overlap.
    Avoids splitting mid-sentence.
    """
    text = normalize_ws(text)
    if not text:
        return []

    sents = sent_split(text, lang)
    if not sents:
        return []

    chunks: List[str] = []
    buf_sents: List[str] = []
    buf_words: List[str] = []

    def buf_tok_len() -> int:
        return _est_tokens_from_words(len(buf_words), lang)

    def flush():
        nonlocal buf_sents, buf_words
        if buf_sents:
            chunks.append(normalize_ws(" ".join(buf_sents)))

        # overlap by keeping tail sentences up to overlap_tokens
        if overlap_tokens > 0 and buf_sents:
            kept_sents: List[str] = []
            kept_words: List[str] = []
            for s in reversed(buf_sents):
                w = s.split()
                if _est_tokens_from_words(len(kept_words) + len(w), lang) <= overlap_tokens:
                    kept_sents.insert(0, s)
                    kept_words = w + kept_words
                else:
                    break
            buf_sents, buf_words = kept_sents, kept_words
        else:
            buf_sents, buf_words = [], []

    for s in sents:
        w = s.split()
        if not w:
            continue

        # If one sentence is too long, hard-slice by words
        if _est_tokens_from_words(len(w), lang) > max_tokens:
            if buf_sents:
                flush()

            step_words = max(40, int(max_tokens / (1.30 if lang == "ne" else 1.33)))
            ov_words = max(10, int(overlap_tokens / (1.30 if lang == "ne" else 1.33)))

            start = 0
            while start < len(w):
                seg = w[start:start + step_words]
                if seg:
                    chunks.append(" ".join(seg).strip())
                start += max(1, step_words - ov_words)
            continue

        # Normal pack
        prospective_words = len(buf_words) + len(w)
        if _est_tokens_from_words(prospective_words, lang) <= max_tokens:
            buf_sents.append(s)
            buf_words.extend(w)
        else:
            # flush when we have enough content; otherwise flush anyway to prevent runaway
            if buf_tok_len() >= min_tokens or len(buf_sents) >= 2:
                flush()
            else:
                flush()
            buf_sents.append(s)
            buf_words.extend(w)

    if buf_sents:
        # allow slightly smaller final chunk if it's meaningful
        if buf_tok_len() >= max(120, int(min_tokens * 0.6)) or len(buf_sents) >= 2:
            chunks.append(normalize_ws(" ".join(buf_sents)))

    # final cleanup (avoid tiny/noisy chunks)
    return [c for c in chunks if c and len(c.split()) >= 30]


In [ ]:
import requests
import tempfile

# PageIndex helpers (Markdown Processing API)
def txt_to_markdown(txt: str, default_title: str) -> str:
    """
    Convert plain text into lightweight markdown to help PageIndex infer structure.
    Conservative: preserves paragraph breaks and promotes heading-like lines.
    """
    lines = [l.rstrip() for l in txt.splitlines()]
    out = [f"# {default_title}", ""]
    buf = []

    def flush_para():
        nonlocal buf
        if buf:
            out.append(" ".join(buf).strip())
            out.append("")
            buf = []

    def looks_like_heading(line: str) -> bool:
        s = line.strip()
        if not s:
            return False
        if re.match(r"^(chapter|section|भाग|अध्याय)\b", s, flags=re.I):
            return True
        if len(s) <= 80 and s.upper() == s and re.search(r"[A-Z]", s):
            return True
        if len(s.split()) <= 10 and len(s) <= 70 and not s.endswith((".", "।", "!", "?")):
            return True
        return False

    for raw in lines:
        line = raw.strip()
        if not line:
            flush_para()
            continue
        if looks_like_heading(line):
            flush_para()
            out.append(f"## {line}")
            out.append("")
        else:
            buf.append(line)

    flush_para()
    return "\n".join(out).strip() + "\n"

def pageindex_markdown_tree(markdown_text: str, api_key: str) -> Dict[str, Any]:
    """
    Calls PageIndex Markdown Processing API and returns JSON.
    Docs: https://docs.pageindex.ai/endpoints
    """
    url = "https://api.pageindex.ai/markdown/"
    headers = {"api_key": api_key}

    with tempfile.NamedTemporaryFile("wb", suffix=".md", delete=True) as f:
        f.write(markdown_text.encode("utf-8"))
        f.flush()
        with open(f.name, "rb") as mdfile:
            resp = requests.post(url, headers=headers, files={"file": mdfile})
    resp.raise_for_status()
    return resp.json()

def flatten_tree_nodes(structure: List[Dict[str, Any]], breadcrumb=None) -> List[Dict[str, Any]]:
    breadcrumb = breadcrumb or []
    out = []
    for node in structure or []:
        title = (node.get("title") or "").strip()
        node_id = node.get("node_id")
        text = node.get("text") or node.get("content")  # tolerate schema drift

        bc = breadcrumb + ([title] if title else [])
        bc_str = " > ".join([b for b in bc if b])

        if isinstance(text, str) and normalize_ws(text):
            out.append({
                "breadcrumb": bc_str,
                "node_title": title,
                "node_id": node_id,
                "text": normalize_ws(text),
            })

        children = node.get("nodes") or []
        if children:
            out.extend(flatten_tree_nodes(children, bc))
    return out


# Build chunks
corpus_chunks: List[Dict[str, Any]] = []
authors = set()
cid = 0

author_dirs = [p for p in glob.glob(os.path.join(CORPUS_ROOT, "*")) if os.path.isdir(p)]
print("Found author folders:", len(author_dirs))

for author_dir in sorted(author_dirs):
    author = os.path.basename(author_dir)
    authors.add(author)

    lang = "ne" if author in NEPALI_AUTHORS else "en"

    txt_files = sorted(glob.glob(os.path.join(author_dir, "*.txt")))
    if not txt_files:
        continue

    for path in txt_files:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        #Convert to markdown to help PageIndex infer structure
        md = txt_to_markdown(text, default_title=os.path.basename(path))

        # PageIndex tree
        tree_json = pageindex_markdown_tree(md, api_key=PAGEINDEX_API_KEY)

        # Flatten nodes into structure-aware passages
        structure = tree_json.get("structure") or tree_json.get("result") or []
        passages = flatten_tree_nodes(structure)

        # Fallback if passages empty (schema change or very small doc)
        if not passages:
            passages = [{"breadcrumb": "", "node_title": "", "node_id": None, "text": normalize_ws(text)}]

        # Chunk each passage with your token-ish sentence packer
        for p in passages:
            base_text = p["text"]
            if p.get("breadcrumb"):
                # breadcrumb gives the embedder strong context anchors
                base_text = f"{p['breadcrumb']}\n{base_text}"

            chunks = chunkify(base_text, lang=lang)

            for chunk in chunks:
                if lang == "ne" and not is_nepali(chunk):
                    continue

                corpus_chunks.append({
                    "chunk_id": f"C{cid:06d}",
                    "author": author,
                    "lang": lang,
                    "path": path,
                    "chunk": chunk,
                    "node_id": p.get("node_id"),
                    "breadcrumb": p.get("breadcrumb"),
                    "node_title": p.get("node_title"),
                })
                cid += 1

print("Authors:", sorted(list(authors)))
print("Total chunks:", len(corpus_chunks))

counts = Counter([c["author"] for c in corpus_chunks])
print("Chunks per author:")
for a, n in counts.most_common():
    print(f"  {a:28s}  {n}")


Found author folders: 16
Authors: ['Austen', 'BPKoirala', 'Balkrishna', 'BhanuBhakta', 'Camus', 'Dickens', 'Dostoevsky', 'Hemingway', 'Kafka', 'LaxmiPrasadDevkota', 'Lekhnath', 'Lovecraft', 'Mainali', 'Orwell', 'Wilde', 'Woolf']
Total chunks: 21185
Chunks per author:
  Dostoevsky                    4528
  Austen                        3440
  Dickens                       3422
  Orwell                        2110
  Hemingway                     1919
  Woolf                         1615
  Wilde                         1121
  Kafka                         999
  Lovecraft                     778
  BPKoirala                     643
  Camus                         445
  LaxmiPrasadDevkota            165


In [ ]:
embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 384

texts = [c["chunk"] for c in corpus_chunks]
if not texts:
    raise RuntimeError("corpus_chunks is empty. Chunking/PageIndex step produced no chunks.")

vecs = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
).astype(np.float32)

print("Embedding shape:", vecs.shape)

dim = vecs.shape[1]

# Use explicit IDs (0..N-1) so alignment with chunks.pkl is always traceable
base_index = faiss.IndexFlatIP(dim)
index = faiss.IndexIDMap2(base_index)

ids = np.arange(len(texts), dtype=np.int64)
index.add_with_ids(vecs, ids)

print("FAISS index size:", index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/663 [00:00<?, ?it/s]

Embedding shape: (21185, 768)
FAISS index size: 21185


In [ ]:
with open(CHUNKS_PKL_PATH, "wb") as f:
    pickle.dump(corpus_chunks, f)

faiss.write_index(index, FAISS_INDEX_PATH)

print("Saved:", CHUNKS_PKL_PATH)
print("Saved:", FAISS_INDEX_PATH)


Saved: /content/model/chunks.pkl
Saved: /content/model/faiss.index


In [ ]:
from google.colab import files
files.download(CHUNKS_PKL_PATH)
files.download(FAISS_INDEX_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>